# 14 · VLA / WA / π0 接口：模仿学习、动作块与世界模型

VLA、World-Action Model 和 π0/openpi 讨论的是“感知条件如何转成可执行动作”，但它们并不自动替代坐标、数据、时延和安全工程。本 notebook 用一个低维 driving state 构造行为克隆策略，再放进带 action chunk 和观测延迟的闭环；最后拟合一个简单 dynamics model，观察 world-model prediction error。

学习目标：

- 用 state、language/route condition 和 action 定义一个可审计的 VLA 接口；
- 区分 behavior cloning 的 open-loop loss 与 closed-loop drift；
- 观察 action chunk、policy period、observation noise 对闭环的影响；
- 理解 WA 的最小接口：预测 action 后果，而不是只生成看似合理的动作。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider

plt.rcParams['figure.figsize'] = (10, 4.5)
dt = 0.1
rng = np.random.default_rng(61)

def expert_action(state):
    distance, relative_speed, lane_error, route_turn = state
    acceleration = np.clip(0.12 * (distance - 14.0) + 0.55 * relative_speed, -3.0, 2.0)
    steering = np.clip(-0.65 * lane_error + 0.35 * route_turn, -1.0, 1.0)
    return np.array([acceleration, steering])

def state_features(states):
    states = np.atleast_2d(states)
    return np.c_[
        np.ones(len(states)),
        states,
        states ** 2,
        states[:, 0:1] * states[:, 1:2],
        states[:, 2:3] * states[:, 3:4],
    ]

states = np.c_[
    rng.uniform(6, 28, 5000),
    rng.uniform(-5, 5, 5000),
    rng.uniform(-2.5, 2.5, 5000),
    rng.uniform(-1, 1, 5000),
]
actions = np.stack([expert_action(state) for state in states])
phi = state_features(states)
weights = np.linalg.solve(phi.T @ phi + 1e-3 * np.eye(phi.shape[1]), phi.T @ actions)

def bc_policy(state):
    return state_features(state) @ weights

train_error = np.mean((bc_policy(states) - actions) ** 2)
print(f'behavior-cloning action MSE: {train_error:.5f}')


In [ ]:
def transition(state, action, lead_speed=8.0):
    distance, relative_speed, lane_error, route_turn = state
    acceleration, steering = np.asarray(action)
    next_distance = np.clip(distance + relative_speed * dt, 0.0, 40.0)
    next_relative_speed = np.clip(relative_speed - acceleration * dt, -10.0, 10.0)
    next_lane_error = lane_error + (steering - 0.15 * route_turn) * dt
    return np.array([next_distance, next_relative_speed, next_lane_error, route_turn])

def rollout(policy_period=1, chunk_horizon=1, observation_noise=0.0, action_noise=0.0, seed=61):
    local = np.random.default_rng(seed)
    state = np.array([22.0, 0.0, 1.2, 0.5])
    states_out, actions_out = [state.copy()], []
    current_chunk = np.zeros((chunk_horizon, 2))
    for step in range(100):
        if step % policy_period == 0:
            observed = state + local.normal(0, observation_noise, size=4)
            first_action = np.asarray(bc_policy(observed)).reshape(2)
            current_chunk = np.tile(first_action, (chunk_horizon, 1))
            current_chunk += local.normal(0, action_noise, size=current_chunk.shape)
        action = current_chunk[min(step % policy_period, len(current_chunk) - 1)]
        state = transition(state, action)
        states_out.append(state.copy())
        actions_out.append(action.copy())
    return np.asarray(states_out), np.asarray(actions_out)

rollout_states, rollout_actions = rollout()
print('final state:', rollout_states[-1].round(3))


In [ ]:
def show_vla(policy_period=2, chunk_horizon=4, observation_noise=0.0, action_noise=0.0):
    states_out, actions_out = rollout(
        policy_period=policy_period,
        chunk_horizon=chunk_horizon,
        observation_noise=observation_noise,
        action_noise=action_noise,
    )
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(states_out[:, 0], label='distance to lead')
    ax[0].plot(states_out[:, 1], label='relative speed')
    ax[0].plot(states_out[:, 2], label='lane error')
    ax[0].axhline(0, color='black', linestyle=':')
    ax[0].set_title('closed-loop state')
    ax[0].set_xlabel('step')
    ax[0].legend()
    ax[1].plot(actions_out[:, 0], label='acceleration')
    ax[1].plot(actions_out[:, 1], label='steering')
    ax[1].set_title('executed action chunk stream')
    ax[1].set_xlabel('step')
    ax[1].legend()
    plt.tight_layout()
    plt.show()
    print('final distance / relative speed / lane error:', states_out[-1, :3].round(3))

interact(
    show_vla,
    policy_period=IntSlider(min=1, max=12, step=1, value=2, description='policy K'),
    chunk_horizon=IntSlider(min=1, max=16, step=1, value=4, description='chunk H'),
    observation_noise=FloatSlider(min=0.0, max=0.8, step=0.05, value=0.0, description='obs noise'),
    action_noise=FloatSlider(min=0.0, max=0.8, step=0.05, value=0.0, description='action noise'),
);


In [ ]:
transition_states, transition_actions, transition_targets = [], [], []
for i in range(3000):
    state = states[i]
    action = actions[i]
    transition_states.append(np.r_[state, action])
    transition_actions.append(np.r_[state, action])
    transition_targets.append(transition(state, action))
transition_states = np.asarray(transition_states)
transition_targets = np.asarray(transition_targets)
dyn_phi = np.c_[np.ones(len(transition_states)), transition_states]
dyn_weights = np.linalg.solve(
    dyn_phi.T @ dyn_phi + 1e-3 * np.eye(dyn_phi.shape[1]),
    dyn_phi.T @ transition_targets,
)
dyn_pred = dyn_phi @ dyn_weights
print('one-step world-model RMSE:', np.sqrt(np.mean((dyn_pred - transition_targets) ** 2)))


### 练习：不要把 action chunk 当成免费加速

- 比较 policy_period=1、4、8 的 closed-loop lane error。
- 观察 observation noise 和 action noise 增大后，BC open-loop MSE 与 closed-loop drift 的差异。
- 用 learned dynamics model 评估两个候选动作的未来状态，加入 distance、lane error 和 action smoothness cost。
- 将 route_turn 视为语言/任务条件的低维代理，思考真实 VLA 如何把视觉、语言、历史状态和 action token 对齐。
- 对照 π0/openpi：这里没有复现其模型规模或数据，只练习相同的接口问题——异构 observation、动作 chunk、实时执行和后果预测。


## 完成标准

- 报告 BC action MSE、closed-loop final state 和不同 policy period 的误差。
- 展示一个 observation stale 导致的失败案例。
- 给出一个 world-model one-step error，并说明多步 rollout error 为什么会累积。
- 明确区分 VLM 的语义条件、VLA 的动作策略、WA 的动作后果建模和 safety monitor 的权限边界。
